# Projet Hadoop — Comprendre l’entraînement du modèle K-Means

Ce notebook explique le projet **pas à pas**, sans supposer de connaissances préalables en machine learning.

## Objectif

Nous avons un dataset de séries TV. Le but est de regrouper automatiquement les séries ayant des profils proches.

Le modèle utilisé est **K-Means**, un algorithme de clustering non supervisé.

> Le modèle ne connaît pas les bonnes réponses à l’avance. Il observe les caractéristiques des séries et crée lui-même des groupes.


## 1. Vue d’ensemble de la pipeline

```text
Fichiers CSV
    ↓
Stockage dans HDFS
    ↓
Nettoyage avec Spark
    ↓
Création de nouvelles variables
    ↓
Assemblage des 26 features
    ↓
Normalisation
    ↓
Entraînement de plusieurs K-Means
    ↓
Comparaison des résultats
    ↓
Choix du modèle final : K = 8, seed = 42
    ↓
Sauvegarde du scaler et du modèle
    ↓
Prédiction des clusters
```

### Vocabulaire essentiel

| Terme | Signification simple |
|---|---|
| Feature | Une information utilisée par le modèle |
| Cluster | Un groupe de séries similaires |
| K | Le nombre de clusters demandé |
| Seed | Une valeur qui permet de reproduire l’entraînement |
| Scaler | Outil qui met les variables à la même échelle |
| Silhouette | Mesure de la séparation des clusters |
| Entraînement | Étape pendant laquelle le modèle apprend les groupes |
| Prédiction | Attribution d’une série à un cluster |


## 2. Illustration simple de K-Means

L’exemple ci-dessous est uniquement pédagogique.

Chaque point représente une série fictive avec deux caractéristiques : popularité et nombre de votes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

points = np.array([
    [1, 1], [1.3, 1.5], [2, 1.2],
    [7, 7], [7.5, 6.5], [8, 7.2],
    [3, 8], [3.5, 8.5], [4, 7.8]
])
clusters = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1], c=clusters)
plt.xlabel("Popularité fictive")
plt.ylabel("Nombre de votes fictif")
plt.title("Exemple simplifié de 3 clusters")
plt.show()


### À retenir

K-Means ne connaît pas les titres des séries. Il voit uniquement des nombres. Deux séries proches numériquement ont plus de chances d’être placées dans le même cluster.


## 3. Initialisation de Spark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round as spark_round
from pyspark.ml.clustering import KMeansModel
from pyspark.ml.feature import StandardScalerModel
import pandas as pd
import json
import matplotlib.pyplot as plt

HDFS_PATH = "hdfs://localhost:9000/user/user/data"

spark = (
    SparkSession.builder
    .appName("Notebook explicatif - Projet séries TV")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark est prêt.")


## 4. Chargement des résultats produits par le modèle

Le fichier `shows_clustered` contient les séries après entraînement. La colonne `cluster_id` correspond au groupe attribué par K-Means.


In [ ]:
df_clustered = spark.read.parquet(f"{HDFS_PATH}/shows_clustered")
print("Nombre total de séries :", df_clustered.count())
df_clustered.printSchema()
df_clustered.show(10, truncate=False)


## 5. Les 26 features utilisées

Le modèle utilise la popularité, le nombre de votes, la décennie, les genres, la longueur de la série, la qualité pondérée des votes et le nombre moyen d’épisodes par saison.


In [ ]:
with open("outputs/feature_cols.json", encoding="utf-8") as f:
    feature_cols = json.load(f)

print("Nombre de features :", len(feature_cols))
feature_cols


### Pourquoi encoder les genres en 0 ou 1 ?

| Série | Comedy | Crime | Drama |
|---|---:|---:|---:|
| Série A | 1 | 0 | 1 |
| Série B | 0 | 1 | 1 |

K-Means ne comprend pas directement les mots. On transforme donc chaque genre en colonne numérique.


## 6. Pourquoi normaliser les données ?

Les colonnes n’ont pas la même échelle. Un genre vaut 0 ou 1, alors que le nombre de votes peut dépasser plusieurs milliers.

Sans normalisation, les colonnes avec les plus grandes valeurs domineraient le calcul. Le `StandardScaler` corrige ce problème.


## 7. Rapport du modèle final


In [ ]:
with open("outputs/model_report.json", encoding="utf-8") as f:
    report = json.load(f)
report


### Résultat retenu

- Modèle : **K-Means**
- K : **8**
- Seed : **42**
- Séries : **53 532**
- Features : **26**
- Silhouette : **environ 0,2707**

La silhouette est moyenne : plusieurs clusters se chevauchent, mais le modèle reste exploitable pour le projet.


## 8. Répartition des séries dans les clusters


In [ ]:
cluster_sizes = df_clustered.groupBy("cluster_id").count().orderBy("cluster_id")
cluster_sizes.show()


In [ ]:
cluster_sizes_pd = cluster_sizes.toPandas()
plt.figure(figsize=(9, 5))
plt.bar(cluster_sizes_pd["cluster_id"].astype(str), cluster_sizes_pd["count"])
plt.xlabel("Cluster")
plt.ylabel("Nombre de séries")
plt.title("Nombre de séries par cluster")
plt.show()


### Interprétation

Le cluster 7 est beaucoup plus grand que les autres. Une grande partie du dataset possède donc un profil assez général, tandis que les autres clusters sont plus spécialisés.


## 9. Profil moyen de chaque cluster


In [ ]:
cluster_summary = (
    df_clustered.groupBy("cluster_id")
    .agg(
        count("*").alias("nb_series"),
        spark_round(avg("popularity"), 2).alias("popularite_moyenne"),
        spark_round(avg("vote_quality"), 2).alias("qualite_vote_moyenne"),
        spark_round(avg("vote_count"), 2).alias("votes_moyens"),
        spark_round(avg("serie_length"), 2).alias("longueur_moyenne"),
        spark_round(avg("content_density"), 2).alias("episodes_par_saison")
    )
    .orderBy("cluster_id")
)
cluster_summary.show(truncate=False)


Ce tableau permet de décrire les clusters : séries très populaires, longues, très votées, ou avec beaucoup d’épisodes par saison.


## 10. Comparaison des entraînements


In [ ]:
results = pd.read_csv("experiments/results.csv")
results


Chaque ligne correspond à un entraînement complet. On compare la silhouette, le coût, la taille du plus petit cluster et celle du plus grand.


In [ ]:
mean_results = results.groupby("k", as_index=False).agg(
    silhouette_moyenne=("silhouette", "mean"),
    cout_moyen=("cout", "mean"),
    cluster_min_moyen=("min_cluster_size", "mean"),
    cluster_max_moyen=("max_cluster_size", "mean")
)
mean_results


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(mean_results["k"], mean_results["silhouette_moyenne"], marker="o")
plt.xlabel("Nombre de clusters K")
plt.ylabel("Silhouette moyenne")
plt.title("Évolution de la silhouette selon K")
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(mean_results["k"], mean_results["cout_moyen"], marker="o")
plt.xlabel("Nombre de clusters K")
plt.ylabel("Coût moyen")
plt.title("Évolution du coût selon K")
plt.show()


### Pourquoi ne pas choisir uniquement la meilleure silhouette ?

Un modèle peut obtenir un bon score mais créer un cluster d’une seule série ou un cluster contenant presque tout le dataset.

`K = 8`, `seed = 42` a été retenu comme compromis : aucun cluster minuscule, résultat lisible, silhouette correcte.


## 11. Vérification de la sauvegarde des modèles


In [ ]:
scaler_model = StandardScalerModel.load(f"{HDFS_PATH}/models/scaler_model")
kmeans_model = KMeansModel.load(f"{HDFS_PATH}/models/kmeans_model")

print("Scaler chargé avec succès.")
print("K-Means chargé avec succès.")
print("Nombre de clusters :", kmeans_model.getK())


Cela prouve que les modèles ont été entraînés, sauvegardés dans HDFS et qu’ils peuvent être rechargés sans nouvel entraînement.


## 12. Cas de test : séries connues


In [ ]:
series_test = ["Lie to Me", "Sherlock", "Breaking Bad", "Friends", "The Mentalist"]

test_results = (
    df_clustered
    .filter(col("name").isin(series_test))
    .select("name", "cluster_id", "popularity", "vote_quality", "vote_count", "serie_length", "content_density")
    .orderBy("name")
)
test_results.show(truncate=False)


Le test vérifie que le modèle attribue bien un cluster, produit un résultat reproductible et peut être appliqué à des exemples connus.


## 13. Profil du cluster de chaque série test


In [ ]:
test_with_profile = (
    test_results.alias("t")
    .join(cluster_summary.alias("c"), col("t.cluster_id") == col("c.cluster_id"), "left")
    .select(
        col("t.name"), col("t.cluster_id"), col("t.popularity"), col("t.vote_count"),
        col("c.nb_series"), col("c.popularite_moyenne"), col("c.votes_moyens"), col("c.longueur_moyenne")
    )
)
test_with_profile.show(truncate=False)


On peut ainsi comparer les valeurs d’une série aux moyennes de son cluster.


## 14. Entraînement, prédiction et rechargement

### Entraînement

```python
model = kmeans.fit(data_scaled)
```

Le modèle calcule les centres des clusters.

### Prédiction

```python
predictions = model.transform(data_scaled)
```

Le modèle attribue chaque série au cluster le plus proche.

### Rechargement

```python
model = KMeansModel.load(...)
```

Le modèle existant est réutilisé sans être entraîné à nouveau.


## 15. Limites du modèle

Le modèle ne comprend pas les scénarios, les personnages, les thèmes psychologiques, le sens des résumés ni les goûts personnels d’un utilisateur.

Il travaille surtout avec des métadonnées numériques et des genres. Cela explique pourquoi certaines recommandations semblent peu pertinentes.


## 16. Conclusion

Le projet montre une pipeline complète :

- données dans HDFS ;
- nettoyage avec Spark ;
- création de 26 features ;
- normalisation ;
- plusieurs entraînements K-Means ;
- comparaison des résultats ;
- sélection de `K = 8`, `seed = 42` ;
- sauvegarde ;
- rechargement ;
- prédiction ;
- analyse avec des cas de test.

Le résultat n’est pas parfait, mais le processus d’entraînement est complet, reproductible et expliqué.


In [ ]:
# À exécuter une seule fois à la fin du notebook
spark.stop()
